# Olink Analyze showcase

This notebook walks through the public `olinkanalyze` functions on the bundled
example datasets:

1. Import NPX (`read_npx`)
2. QC (`check_npx`, `extract_check_log`, `clean_npx`)
3. Plots (`olink_dist_plot`, `olink_qc_plot`, `olink_volcano_plot`)
4. Bridge normalization (`olink_normalization`)
5. Statistics (`olink_ttest`)

Install the package in this repo first (`pip install -e .`), then run the
cells in order.

## Setup

In [ ]:
%matplotlib inline

import pandas as pd

import olinkanalyze as oa
from olinkanalyze.data import load_example_csv

pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 120)

print("olinkanalyze", oa.__version__)
print("functions:", ", ".join(name for name in oa.__all__ if name != "__version__"))

## Example datasets

`npx_data1` and `npx_data2` are long-format NPX tables shipped with the
package. They share overlapping (bridge) samples so they can be normalized
together.

In [ ]:
npx_data1 = oa.npx_data1
npx_data2 = oa.npx_data2

print("npx_data1:", npx_data1.shape)
print("npx_data2:", npx_data2.shape)
npx_data1.head()

In [ ]:
summary = pd.DataFrame(
    {
        "npx_data1": [
            npx_data1["SampleID"].nunique(),
            npx_data1["OlinkID"].nunique(),
            npx_data1["Panel"].nunique(),
            npx_data1["Project"].nunique(),
        ],
        "npx_data2": [
            npx_data2["SampleID"].nunique(),
            npx_data2["OlinkID"].nunique(),
            npx_data2["Panel"].nunique(),
            npx_data2["Project"].nunique(),
        ],
    },
    index=["samples", "assays", "panels", "projects"],
)
summary

## `read_npx`

`read_npx` imports long-format CSV/TSV, Excel, or Parquet. It runs
`check_npx` and stores the log on `DataFrame.attrs`. `read_NPX` is the same
function under the historical alias.

In [ ]:
csv_path = load_example_csv()
raw = oa.read_npx(csv_path)

print(csv_path.name)
print("shape:", raw.shape)
print("columns:", list(raw.columns))
raw.head()

## `check_npx` and `extract_check_log`

`check_npx` returns a dictionary of QC findings without changing the data.
`extract_check_log` reads the copy that `read_npx` / `clean_npx` attach to
the frame.

In [ ]:
check_log = oa.check_npx(npx_data1)
attached = oa.extract_check_log(raw)


def summarize_check_log(log: dict) -> pd.DataFrame:
    rows = []
    for key, value in log.items():
        if key == "col_names":
            detail = f"{len(value)} mapped columns"
        elif isinstance(value, pd.DataFrame):
            detail = f"{len(value)} row(s)"
        elif isinstance(value, list):
            preview = ", ".join(map(str, value[:5]))
            extra = "" if len(value) <= 5 else ", ..."
            detail = f"{len(value)}: {preview}{extra}" if value else "0"
        else:
            detail = str(value)
        rows.append({"check": key, "result": detail})
    return pd.DataFrame(rows)


print("Keys on the attached log:", list(attached.keys()) if attached else None)
summarize_check_log(check_log)

## `clean_npx`

Drop control samples, invalid OlinkIDs, all-NA assays, QC/assay warnings,
and duplicate sample IDs before analysis. Pass `verbose=True` to see which
filters fire.

In [ ]:
npx_clean = oa.clean_npx(npx_data1, check_log=check_log)

print("rows before:", len(npx_data1))
print("rows after:", len(npx_clean))
print("sample types remaining:", sorted(npx_clean["SampleType"].dropna().unique()))
npx_clean.head()

## `olink_dist_plot`

NPX boxplots per sample, faceted by panel. Points are colored by
`QC_Warning` by default.

In [ ]:
dist_fig = oa.olink_dist_plot(npx_clean)
dist_fig

## `olink_qc_plot`

Sample-level IQR versus median NPX. Use this to spot outlier samples.

In [ ]:
qc_fig = oa.olink_qc_plot(npx_clean, label_outliers=True)
qc_fig

## `olink_normalization` (bridge)

Shift the non-reference project by the per-assay median NPX difference on
overlapping samples. Only bridge normalization is implemented.

In [ ]:
bridge_samples = sorted(
    set(npx_data1["SampleID"]).intersection(npx_data2["SampleID"])
)
bridge_samples = [sample for sample in bridge_samples if "CONTROL" not in str(sample)]
print(f"{len(bridge_samples)} overlapping study samples")

normalized = oa.olink_normalization(
    df1=npx_data1,
    df2=npx_data2,
    overlapping_samples_df1=bridge_samples,
    df1_project_nr="20200001",
    df2_project_nr="20200002",
    reference_project="20200001",
)

print("normalized shape:", normalized.shape)
print("projects:", sorted(normalized["Project"].astype(str).unique()))
normalized[["SampleID", "OlinkID", "NPX", "Project", "Adj_factor"]].head()

`olink_normalization_bridge` is the same method with explicit project
argument names:

In [ ]:
bridged = oa.olink_normalization_bridge(
    project_1_df=npx_data1,
    project_2_df=npx_data2,
    overlapping_samples_df1=bridge_samples,
    project_1_name="20200001",
    project_2_name="20200002",
    reference_project="20200001",
)
print("same shape:", bridged.shape == normalized.shape)
print("Adj_factor range:", float(bridged["Adj_factor"].min()), "to", float(bridged["Adj_factor"].max()))

## `olink_ttest`

Welch two-sample t-test per assay. `variable` must have exactly two levels.
Multiple-testing correction is Benjamini-Hochberg FDR.

A paired test is available with `pair_id`.

In [ ]:
ttest_results = oa.olink_ttest(npx_clean, variable="Treatment")
ttest_results.head(10)

In [ ]:
print(ttest_results["Threshold"].value_counts().to_string())
print()
print(
    ttest_results.loc[
        ttest_results["Threshold"] == "Significant",
        ["Assay", "OlinkID", "estimate", "p.value", "Adjusted_pval"],
    ].head(8)
)

paired = npx_clean.loc[npx_clean["Time"].isin(["Baseline", "Week.6"])]
paired_ttest = oa.olink_ttest(paired, variable="Time", pair_id="Subject")
paired_ttest[["Assay", "OlinkID", "method", "p.value", "Threshold"]].head()

## `olink_volcano_plot`

`-log10(p-value)` versus the t-test estimate. Pass `olinkid_list` to label
specific assays; otherwise significant assays are labeled.

In [ ]:
volcano_fig = oa.olink_volcano_plot(
    ttest_results,
    olinkid_list=ttest_results["OlinkID"].head(8),
)
volcano_fig

## `olink_pal` and `set_plot_theme`

The plotting helpers use the Olink palette. You can also pull those colors
for your own matplotlib / seaborn charts.

In [ ]:
import matplotlib.pyplot as plt

palette = oa.olink_pal()
fig, ax = plt.subplots(figsize=(7, 1.4))
ax.bar(range(len(palette)), [1] * len(palette), color=palette)
ax.set_xticks(range(len(palette)))
ax.set_xticklabels([f"C{i+1}" for i in range(len(palette))], fontsize=9)
ax.set_yticks([])
ax.set_title("olink_pal()")
oa.set_plot_theme(ax)
fig

## Function map

| Function | Role |
| --- | --- |
| `read_npx` / `read_NPX` | Import long-format NPX (CSV/TSV, Excel, Parquet) |
| `check_npx` | Build a QC `check_log` |
| `extract_check_log` | Read the log stored on `df.attrs` |
| `clean_npx` | Filter controls, warnings, and invalid IDs |
| `olink_dist_plot` | NPX distribution by sample |
| `olink_qc_plot` | Sample IQR vs median |
| `olink_normalization` / `olink_normalization_bridge` | Bridge (reference-sample) normalization |
| `olink_ttest` | Per-assay Welch or paired t-test + BH FDR |
| `olink_volcano_plot` | Volcano from `olink_ttest` results |
| `olink_pal` / `set_plot_theme` | Colors and plot styling |

Wide qPCR exports, ZIP archives, mixed-model statistics, and cross-product
normalization are not implemented yet.